# Update as of: March 9, 2026


## Author: Sandy G Cabanes
## Role: Sole Contributor (Volunteer)
## Date: March 2026
## Response Dates: September 2025 to March 2026
## Licensed Under MIT

In [1]:
# 0. Imports & Config
# ~~~~~~~~~~~~~~~~~~~~~~~~~
import pandas as pd
import unicodedata
from pathlib import Path
import numpy as np
import csv # used in apply_lookup function

In [22]:
# 0. Create Directories
# Store all lookup files for single-response columns and exploded multi-response csv files
lookup_dir = Path("lookup_dir")
lookup_dir.mkdir(exist_ok=True)

# Store all parquet files
parquet_outputs_dir = Path("parquet_outputs_dir")
parquet_outputs_dir.mkdir(exist_ok=True)

# Store all the outputs as csv so cleaning can be split into modules
csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)

# Copy of all the final outputs for data model
final_outputs_dir = Path("final_outputs_dir")
final_outputs_dir.mkdir(exist_ok=True)

# Store preliminary unique values
unique_dir = Path("unique_dir")
unique_dir.mkdir(exist_ok=True)

# Store all csv and map files using country and city columns
location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)


##  Lineage

- Raw file  
  - csv_raw.csv -> df_raw
  - df_raw (cleaned) <- df_raw

- Single‑response family  
  - df_single_no_grps <- df_raw[single_cols_no_grps].copy()  
  - df_single_with_grps <- df_single_no_grps.copy()  
  - df_possible_dups <- df_likely_dups that have a score == 1
  - df_sim_raw <- df_single_no_grps.copy()
  - df_sim_sorted <- df_sim_raw

- Multi‑response family  
  - csv_raw.csv -> df_raw_multi -> clean MS Fabric, clean and standardize None responses
  - df_multi <- df_raw_multi[["resp_id"] + multi_response_cols].copy()
  - datamasters_count <- df_multi_copy
  - df_exploded = explode_and_lookup(df_raw, col) -> saved to csv as {col}_exploded.csv
  - df_exploded_cleaned 
  - df_merged_dups = df_possible_duplicates.merge()

- Location family  
  - df_loc_new <- df_single_with_groups
  - df_ph_grp, df_non_ph_grp <- df_loc_new 
  - overseas <- df_ph_grp 
  - df_ph_geo, df_non_ph_geo <- df_ph_grp, df_non_ph_grp
  - df_ph_geo_clean <- df_ph_geo
  - df_non_ph_geo_clean <- df_non_ph_geo
  - df_all_geo_clean <- df_ph_geo_clean, df_non_ph_geo_clean

-  Data mart family  
    - survey2026.duckdb
    - survey.sqlite
    - for_tableau.xls


# REMOVE TEST RUN FIRST!!!

In [11]:
# 1. Load Raw Data and Rename Columns (Raw -> Transformed)
# IMPORTANT: Remove the test runs first before saving as csv_raw.csv
df_raw= pd.read_csv(
    "csv_raw.csv",
    sep=",",
    quotechar='"',
    engine="python",
    encoding="utf-8-sig"
)

df_raw = df_raw.drop(columns = ['Timestamp', 'Email (Optional)  ', 'End of survey'])

In [ ]:
# 1a. Check the headers
headers_raw = df_raw.columns.to_list()
headers_raw

In [12]:
# 1b. Rename Columns (Raw → Transformed)
# columns = rename_map dictionary

rename_map = {
    "Country of Residence, current" : "country", #new
    "City of Residence, current": "city",
    "PHILIPPINE City of Residence, current (Please scroll to see all options.  If municipality, please write in Other - ______.   If not in the Philippines, please choose 'Not applicable' .)" : "city_ph",
    'NON-PHILIPPINE City of Residence (Choose Not Applicable if you are currently in the Philippines)' : "city_non_ph",
    "Gender": "gender",
    "Age (Please enter a number)": "age",
    "Latest education status": "educstat",
    "Do you have a Computer or Data-Related degree? ": "have_comp_degree", #new
    "If you have a Computer-Related degree, or Data-Related degree,  please indicate the degree below. \n(Otherwise, please choose Not Applicable, first button.)" : "comp_related_degree", #new
    "Choose the digital learning platforms or tools you are currently using for data-related learning:": "digitools",
    "What best describes the CURRENT STAGE of your data career?" : "careerstg",
    "Thinking of your most recent job search, which platform or method gave you the most success?": "successmethod",
    "Type of employer (Please select one option that best describes your current employment situation.)" : "employertype",
    "What Industry currently working in:": "industry",
    "Monthly Salary Range (Or monthly income from main source) If paid per project, please approximate monthly equivalent." : "salary",
    "How long have you been in the current salary range or monthly income range?" : "how_long_in_salary",  #new
    "What is your current Type of Work?": "typework", #new 
    "What is your current Work Site situation?": "sitework",
    "If working and in the Philippines, what is your working shift schedule most of the time? (If not in the Philippines, choose Not Applicable.)" : "shift_schedule",  #new
    "What best describes MAJORITY of your day-to-day role? (More than 50% of your role)": "datarole", #uncategorized in 2024
    "What other descriptions comprise the REST of your role? (Click all that apply)": "restofrole", #uncategorized in 2024
    "Thinking of the past 12 months in terms of side gigs or side hustles, which one applies to you?" : "side_gig", #new
    "On a scale of 1 to 10, how satisfied are you with your current data-related role or data career?  (1 - Not Satisfied At All, 10 - Extremely Satisfied)" : "satisfaction", #new
    "What is the size of your Data Team?": "sizeteam",
    "What are the data INGESTION tools you currently use? ": "ingestion",
    "What are the data TRANSFORMATION tools you currently use?  ": "transform",
    "What are the DATA LAKES OR OBJECT STORAGE SERVICES you currently use?" : "storage", #new
    "What are the data WAREHOUSES or RELATIONAL DATABASES that you currently use?   ": "warehs",
    "What are the data ORCHESTRATION tools you currently use?  ": "orchest",
    "What are the BUSINESS INTELLIGENCE tools you currently use? ": "busint",
    "What are the CLOUD-NATIVE DATA PLATFORMS you currently use?" : "cloudplat",
    "Which of the following \nNON-CODING-LANGUAGE DATA TOOLS OR PRODUCTIVITY TOOLS \ndo you use on a regular basis? Choose all that apply.": "generaltools",
    "Which of the following CODING LANGUAGES OR TOOLS do you use on a regular basis? Choose all that apply.": "whatused",
    "Thinking of WORK, do you currently use AI in your work?": "ai_work", #new
    "Thinking of STUDIES or UPSKILLING, do you currently use AI for study?": "ai_study", #new
    "If you answered Yes to use of AI at work or for study, please choose which AI tools you currently use.  (Choose Not Applicable if you do not use AI.)" : "ai_tools", # revised
    "What hardware do you currently use for data?": "hardware",
    "Which of the following DEP initiatives and channels are you aware of? Choose all that apply." : "dep_init", #revised
    "Any specific tasks, skills, knowledge or resources you are willing to contribute to the group as of today?": "volunteer",
    "Any specific needs you are trying to address by joining DEP Facebook group?" : "spneeds",
    "Thinking of data-related communities, what other Facebook communities do you follow?": "otherfb",
    'In the Past 12 months, what was the most recent DEP or data-related meet-up, hackathon, convention, that you attended IN PERSON? (Maximum 50 characters.  If none, it\'s ok, you can type "None")': "attended_inperson", #new
    'In the Past 12 months, what was the most recent DEP or data-related activity that you attended ONLINE? (E.g. discord event, free webinar, etc.   Maximum 50 characters.  If none, it\'s ok, you can type "None")' : "attended_online" #new
}


df_raw.rename(columns=rename_map, inplace=True)

In [ ]:
#1c. Check mapped headers
headers_revised= df_raw.columns.to_list()
headers_revised

In [13]:
#1d. Column data types
df_raw.info()

# Result as of March 5: n1732 total, n763 working

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1732 entries, 0 to 1731
Data columns (total 42 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   country              1732 non-null   object 
 1   city_ph              1732 non-null   object 
 2   city_non_ph          1715 non-null   object 
 3   gender               1732 non-null   object 
 4   age                  1732 non-null   int64  
 5   educstat             1732 non-null   object 
 6   have_comp_degree     1732 non-null   object 
 7   comp_related_degree  1729 non-null   object 
 8   digitools            1731 non-null   object 
 9   careerstg            1732 non-null   object 
 10  successmethod        763 non-null    object 
 11  employertype         763 non-null    object 
 12  industry             763 non-null    object 
 13  salary               763 non-null    object 
 14  how_long_in_salary   763 non-null    object 
 15  typework             763 non-null    o

In [ ]:
# 2. First Global Cleaning (Inline)
# Fillna, normalize font, strip, collapse spaces, title-case specific columns, 
# Replace Other with blanks, replace 0 age with min age, cap at 95

# Fillna, normalize font ex. ñ, strip, split
for col in df_raw.select_dtypes(include="object").columns:
    # Normalize, strip, collapse spaces
    df_raw[col] = (
        df_raw[col]
        .fillna("Not Applicable")
        .astype(str)
        .str.replace(r'[\u2010-\u2015]', '-', regex=True)
        .str.replace("â€“", "-", regex=False)
        .apply(lambda v: " ".join(
            unicodedata.normalize("NFKC", v.replace("ñ", "n")).strip().split()
        ))
    )
    # Title-case specific columns (For UK, KSA, etc. handle in the lookup portion to revert)
    if col in ["country", "city_ph", "city_non_ph", "datarole", "rest_of_role", "industry"]:
        df_raw[col] = df_raw[col].str.title()

# Replace strings that start with 'Other' with blanks
for col in df_raw.select_dtypes(include="object").columns:
     df_raw[col] = df_raw[col].apply(lambda v: "" if v.startswith("Other") else v)

# Replace age of 0 with min age, and cap at 95
valid_min_age = df_raw.loc[df_raw["age"] > 0, "age"].min()
df_raw["age"] = df_raw["age"].replace(0, valid_min_age)

max_age = df_raw["age"].max()
df_raw["age"] = df_raw["age"].apply(lambda x: max_age if x > 95 else x)

print ("First global cleaning done.")


First global cleaning done.


In [ ]:
#2a. Check after first global cleaning Or Open in Data Wrangler
df_raw.head()

In [16]:
# 3. Add resp_id and resp_id_rand to df_raw after first global cleaning
# Add resp_id
if "resp_id" not in df_raw.columns:
    df_raw.insert(0, "resp_id", range(1, len(df_raw) + 1))

# Add resp_id_rand    
import secrets

def generate_ids(n):
    ids = set()
    # Running the loop until the set contains the required number of unique IDs
    while len(ids) < n:
        # Creating a random 6-character hex string
        new_id = f"2025-{secrets.token_hex(3)}"
        ids.add(new_id)
    return list(ids)

# Assigning the unique list to the dataframe column
df_raw['resp_id_rand'] = generate_ids(len(df_raw))

In [ ]:
#3a. Check in Data Wrangler → export as df_raw.parquet
df_raw.head()

In [ ]:
#3b. Count empty strings and null values or isna and print only those with values
# empty_counts series: Count empty strings - this will include the "" replacements above
empty_counts = (df_raw == "").sum()
empty_columns_only = empty_counts[empty_counts > 0] #only the ones with values
print("\n\nEmpty strings", "~"*20)
if not empty_columns_only.empty:
    print(empty_columns_only)
else:
    print("No empty strings found in any column.")

# nan_counts series: Count NaN / Null values -- expect satisfaction to be total n minus working n
nan_counts = df_raw.isna().sum()
nan_columns_only = nan_counts[nan_counts > 0] #only the ones with values
print("\n\nNaN / Null values", "~"*20)
if not nan_columns_only.empty:
    print(nan_columns_only)
else:
    print("No NaN values found in any column.")

# Calculate NaN and empty cells 
combined_counts = df_raw.apply(
    lambda col: (
        col.isna() | 
        (col.astype(str).str.strip() == "")
    ).sum()
)
combined_only  = combined_counts[combined_counts > 0] #only the ones with values
print("\n\nCombined NaN or blanks", "~"*20)
if not combined_only.empty:
    print(combined_only)

# Result as of March 5:datarole (30), restofrole(29), satisfaction (969), transform(2)




Empty strings ~~~~~~~~~~~~~~~~~~~~
datarole      30
restofrole    29
transform      2
dtype: int64


NaN / Null values ~~~~~~~~~~~~~~~~~~~~
satisfaction    969
dtype: int64


Combined NaN or blanks ~~~~~~~~~~~~~~~~~~~~
datarole         30
restofrole       29
satisfaction    969
transform         2
dtype: int64


In [33]:
# 3c. Replace all empty strings and null values with "Not Applicable"
# Clean whitespace and empty strings
df_raw = df_raw.replace(r'^\s*$', "Not Applicable", regex=True)

# Clean technical Nulls (NaN)
df_raw = df_raw.fillna("Not Applicable")

# Special treatment for satisfaction, that requires numeric values
df_raw["satisfaction"] = df_raw["satisfaction"].apply(lambda x: 999 if x == "Not Applicable" else x)

In [ ]:
df_raw

In [ ]:
# If country is not Philippines, check if city_non_ph is Not Applicable or a Philippine city

df_nonph =df_raw[df_raw['country'] != 'Philippines'][['resp_id', 'country', 'city_non_ph']]
nonphlist =sorted(list(df_nonph['city_non_ph'].unique()))
print("Before cleaning non ph cities", "~"*20)
print (nonphlist)

# Ok, nonph cities are not Philippine cities.  

# Change the 'Not Applicable' to 'Prefer Not To Say'
filter_nonph = (df_raw['country'] != 'Philippines') & (df_raw['city_non_ph'] == 'Not Applicable')
df_raw.loc[filter_nonph, 'city_non_ph'] = 'Prefer Not To Say'

# Check again
df_nonph =df_raw[df_raw['country'] != 'Philippines'][['resp_id', 'country', 'city_non_ph']]
nonphlist =sorted(list(df_nonph['city_non_ph'].unique()))
print("After cleaning non ph cities", "~"*20)
print (nonphlist)

Before cleaning non ph cities ~~~~~~~~~~~~~~~~~~~~
['Abu Dhabi, United Arab Emirates', 'Accra', 'Alexandria', 'Amman', 'Bangkok', 'Barishal', 'Benin City', 'Cairo', 'Cape Town', 'Cerro De Pasco', 'Cheongju City', 'Diplo', 'Douala', 'Dubai', 'Ho Chi Minh', 'Jeddah', 'Kansas', 'Kathmandu', 'Kinshasa', 'Ksa', 'Kuala Lumpur', 'Lagos', 'Lagos State', 'London', 'Newcastle', 'Not Applicable', 'Oslo', 'Pasco', 'Phnom Penh', 'Prefer Not To Say', 'Regina', 'Riyadh', "Sana'A", 'Soroti City Uganda', 'Sydney', 'Uae', 'Ughelli']
After cleaning non ph cities ~~~~~~~~~~~~~~~~~~~~
['Abu Dhabi, United Arab Emirates', 'Accra', 'Alexandria', 'Amman', 'Bangkok', 'Barishal', 'Benin City', 'Cairo', 'Cape Town', 'Cerro De Pasco', 'Cheongju City', 'Diplo', 'Douala', 'Dubai', 'Ho Chi Minh', 'Jeddah', 'Kansas', 'Kathmandu', 'Kinshasa', 'Ksa', 'Kuala Lumpur', 'Lagos', 'Lagos State', 'London', 'Newcastle', 'Oslo', 'Pasco', 'Phnom Penh', 'Prefer Not To Say', 'Regina', 'Riyadh', "Sana'A", 'Soroti City Uganda', 'Sydn

In [ ]:
# Open in Data Wrangler
df_nonph

In [ ]:
# If country is Philippines, check 
df_ph = df_raw[df_raw['country'] == 'Philippines'][['resp_id', 'country', 'city_ph', 'city_non_ph']]
df_ph # Open in Data Wrangler

In [35]:
# 3c. Export df_raw to parquet_outputs_dir
df_raw.to_parquet(parquet_outputs_dir/ "df_raw.parquet", index=False)
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("df_raw.parquet created. Global cleaning done.")

Date and Time: 2026-03-09 16:22:29
df_raw.parquet created. Global cleaning done.


# Handle single-response columns first
- single_cols_no_grps -> single_cols_non_numeric -> write unique_dir / "unique_single.txt"
- create starter csv for lookup with "raw" column -> copy to lookup_dir -> manually add "clean" column 

In [49]:
# 4a. Define single response columns for lookups
# This section uses df_single_no_grps - 23 columns
# Later this will be split into three types of cleaning: direct, special and manual
single_cols_no_grps = [
    "resp_id", "country", "city_ph", "city_non_ph", "gender", "age", "educstat", "have_comp_degree", "comp_related_degree", "careerstg", "employertype", "industry", "salary", "how_long_in_salary", "typework", "sitework", "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam", "ai_work", "ai_study", "resp_id_rand"
]

df_single_no_grps = df_raw[single_cols_no_grps].copy()

In [ ]:
# 4a2. Print out to terminal all unique values
# Check unique values first except resp_id  
# Write all unique values to a single file except resp_id, age

single_columns_non_numeric = [col for col in df_single_no_grps.columns if col not in ['resp_id', 'age', 'resp_id_rand', 'satisfaction']]

with open(unique_dir / "unique_single.txt", "w", encoding="utf-8") as f:
    for col in single_columns_non_numeric:
        f.write(f"{col}\n\n")
        for val in df_single_no_grps[col].dropna().unique():
            f.write(f"{val}\n")
        f.write("\n")
        
print("unique_single.txt created.  Inspect unique values.")
with open(unique_dir / "unique_single.txt", "r", encoding="utf-8") as f:
    print(f.read())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

unique_single.txt created.  Inspect unique values.
country

Philippines
United Kingdom
Norway
Cameroon
Cambodia
Thailand
Pakistan
United Arab Emirates
Canada
Nepal
United States
Uae
Nigeria
Jordan
Egypt
Ksa
South Africa
Uganda
Congo
South Korea
Malaysia
Bangladesh
Yemen
Saudi Arabia
Peru
Perú
Kenya
Algeria
Vietnam
Ghana
India
Spain

city_ph

Pampanga
Binan
Mandaluyong
San Quintin
Taguig
Not Applicable
Aleosan
Dasmarinas
Muntinlupa
Quezon City
Pasig
Caloocan
Davao
Paranaque
Manila
Bacolod
Makati
Batangas
Zamboanga
Roxas
Las Pinas
La Pinas
Ozamiz City
Cavite
Calamba
Binalonan
Bulacan
Imus
Cebu
Tanza
Baliwag
Lake Sebu, South Cotabato
La Union
Camarines Sur
Calbayog Samar
Cagayan De Oro
Calbayog City
Laguna
Roxas City
San Fernando
Davao Del Norte
Surigao City
San Pablo
Bukidnon
Tarlac
Rizal
Iloilo
Valencia City Bukidnon
Cabuyao
El Salvador City
Malabon City
Negros Occidental
Isabela
San Jose
Valenzuela
Pasay
Cainta
Norzagaray
Albay
Mabalacat
Antipolo
General Mariano Alvarez
Marikina
Taytay

In [ ]:
# 4b. Convert unique values to starter csv files for lookup and store in unique_dir 
# -> copy and edit into lookup_dir as final lookup csv 
# -> unique_dir -> lookup_dir after manual transformation

single_columns_non_numeric = [
    col for col in df_single_no_grps.columns
    if col not in ["resp_id", "age", "resp_id_rand", "satisfaction"]
]

for col in single_columns_non_numeric:
    uniques = (
        df_single_no_grps[col]
        .dropna()
        .astype(str)
        .unique()
    )
    uniques_sorted = sorted(uniques)

    out_df = pd.DataFrame({"raw": uniques_sorted})
    out_path = unique_dir / f"{col}.csv"
    out_df.to_csv(out_path, index=False, encoding="utf-8")

print("~"*80)
print("Per-column unique CSVs created in unique_dir.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Per-column unique CSVs created in unique_dir.
Date and Time:  2026-03-09 16:47:38


In [55]:
# 4c. Create temp lookup csv files (staging column) in lookup_dir using the raw column values as clean_temp
# Three types of edits: 1. no edits (direct clean), 2. special edits, 3. manual edits (the rest)

# List of no edit files - type 1
direct_clean_csv_files = ["ai_study_lookup.csv", "ai_work_lookup.csv", "careerstg_lookup.csv", "datarole_lookup.csv", "educstat_lookup.csv", "employertype_lookup.csv", "gender_lookup.csv", "industry_lookup.csv","satisfaction_lookup.csv", "shift_schedule_lookup.csv","sitework_lookup.csv","sizeteam_lookup.csv", "typework_lookup.csv"]

# List of special edit files - type 2
specific_csv_lookup_files = ["have_comp_degree_lookup.csv", "side_gig_lookup.csv", "salary_lookup.csv", "how_long_in_salary_lookup.csv"]

# Create temp lookup csv files in lookup_dir
for file in unique_dir.glob("*.csv"):
    df = pd.read_csv(file, dtype=str)
    
    # Copy the raw column to "clean_temp" staging column
    df["clean_temp"] = df["raw"]
    
    # Generate the target filename as lookup
    lookup_filename = f"{file.stem}_lookup.csv"
    
    # Directly convert "clean_temp" to "clean" for direct clean →  Type 1 edit
    if lookup_filename in direct_clean_csv_files:
        # Renaming the column to 'clean' for matches
        df.rename(columns={"clean_temp": "clean"}, inplace=True)

    # Exporting the file to the output directory
    out_path = lookup_dir / lookup_filename
    df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Created: {out_path}")


Created: lookup_dir\ai_study_lookup.csv
Created: lookup_dir\ai_work_lookup.csv
Created: lookup_dir\careerstg_lookup.csv
Created: lookup_dir\city_non_ph_lookup.csv
Created: lookup_dir\city_ph_lookup.csv
Created: lookup_dir\comp_related_degree_lookup.csv
Created: lookup_dir\country_lookup.csv
Created: lookup_dir\datarole_lookup.csv
Created: lookup_dir\educstat_lookup.csv
Created: lookup_dir\employertype_lookup.csv
Created: lookup_dir\gender_lookup.csv
Created: lookup_dir\have_comp_degree_lookup.csv
Created: lookup_dir\how_long_in_salary_lookup.csv
Created: lookup_dir\industry_lookup.csv
Created: lookup_dir\salary_lookup.csv
Created: lookup_dir\satisfaction_lookup.csv
Created: lookup_dir\shift_schedule_lookup.csv
Created: lookup_dir\side_gig_lookup.csv
Created: lookup_dir\sitework_lookup.csv
Created: lookup_dir\sizeteam_lookup.csv
Created: lookup_dir\typework_lookup.csv


In [56]:
# 4d. Special lookup files given different wording vs. labels for charts
# have_comp_degree, side_gig, salary, how_long_in_salary
# Create map → apply map function & drop temp → export

import pandas as pd


# Define all maps → feeds into dictionary → replace "clean_temp" in lookup
map_comp_degree = {
    "No computer-related or data-related degree": "Not have",
    "Yes, I have a computer-related or data-related degree": "Have comp or data degree"
}

map_side_gig = {
    "I HAD at least one side gig in the past 12 months": "Side gig p12m",
    "I did NOT have side gigs in the past 12 months": "No side gig p12m"
}

map_salary = {
    "100,001 to 125,000":"100k+ to 125k",
    "125,001 to 250,000":"125k+ to 250k",
    "15,000 and below":"15k and below",
    "15,001 to 25,000":"15k+ to 25k",
    "25,001 to 35,000":"25k+ to 35k",
    "250,001 and above":"250k and above",
    "35,001 to 45,000":"35k+ to 45k",
    "45,001 to 55,000":"45k+ to 55k",
    "55,001 to 65,000":"55k+ to 65k",
    "65,001 to 75,000":"65k+ to 75k",
    "75,001 to 85,000":"75k+ to 85k",
    "85,001 to 95,000":"85k+ to 95k",
    "95,001 to 100,000":"95k+ to 100k"
}

map_salary_duration = {
    "3 months or less": "a. 3mos or less",
    "More than 3 to 6 months": "b. >3 to 6mos",
    "More than 6 months less than 1 year": "c. >6mos to 1yr",
    "More than 1 year to 2 years": "d. >1yr to 2yrs",
    "More than 2 years to 5 years": "e. >2 yrs to 5yrs",
    "More than 5 years to 10 years": "f. >5 yrs to 10yrs",
    "More than 10 years": "g. >10yrs",
    "None of the above": "h. Not Applicable",
    "Not Applicable": "Not Applicable"
}

# Dictionary for all special mappings 
all_mappings = {
    "have_comp_degree": map_comp_degree,
    "side_gig": map_side_gig,
    "salary": map_salary,
    "how_long_in_salary": map_salary_duration
}

# Function apply_special_map → lookup files
def apply_special_map(col_name, mapping_dict, lookup_dir):
    """
    Same method, different mappings. Reads, maps, cleans, and exports lookup CSVs.
    Uses 'clean_temp' as the common source column.
    """
    file_path = lookup_dir / f"{col_name}_lookup.csv"
    df = pd.read_csv(file_path, dtype=str)    
    df["clean"] = df["clean_temp"].map(mapping_dict).fillna(df["clean_temp"])
    
    # Remove the staging column and overwrite the source file
    df.drop(columns=["clean_temp"], inplace=True)
    df.to_csv(file_path, index=False)

# Apply function to special columns
for col, mapping in all_mappings.items():
    apply_special_map(col, mapping, lookup_dir)


In [ ]:
# 4d. Identify files to manually edit → Type 3 edit

# This is unavoidable. Some columns really need human input.  Attempt to use AI resulted in missed important items (ex. BS Business Administration ← failed to convert to  BSBA Major Unspecified, Two year computer course ← failed to convert to Associate, and more)

# Initialize {set} object to contain all file names in lookup dir using file paths
all_files_in_lookup_dir_set = {f"{file.stem}_lookup.csv" for file in unique_dir.glob("*.csv")}

# Subtract direct clean csv files and special csv lookup files.
# Use the set difference to remove the clean files
manual_edit_lookup_files = sorted(list(all_files_in_lookup_dir_set - set(direct_clean_csv_files) - set(specific_csv_lookup_files)))

print("Files to manually edit:")
print(manual_edit_lookup_files)
print("REMOVE ALL COMMAS AND NON LETTER CHARACTERS, OPTIONS THAT CAN BE CONVERGED TO ONE, WRONG GRAMMAR")
print("AND THEN COPY AND PASTE INTO THE CORRECT FILENAME IN THE MANUAL_EDIT FOLDER")

Files to manually edit:
['city_non_ph_lookup.csv', 'city_ph_lookup.csv', 'comp_related_degree_lookup.csv', 'country_lookup.csv']
REMOVE ALL COMMAS AND NON LETTER CHARACTERS, OPTIONS THAT CAN BE CONVERGED TO ONE, WRONG GRAMMAR
AND THEN COPY AND PASTE INTO THE CORRECT FILENAME IN THE MANUAL_EDIT FOLDER


## REMINDER: COPY AND PASTE MANUAL EDIT LOOKUP FILES IN A MANUAL_EDIT SUBFOLDER NOW.

In [59]:
# Lookup files -- Add Not Applicable to both raw and clean columns if not yet present
from pathlib import Path
import pandas as pd

# Define the directory containing your data files
lookup_dir = Path("lookup_dir")

for file in lookup_dir.glob("*.csv"):
    # Load the data into a structured table format
    # We use 'latin-1' to ensure compatibility with various data sources
    df = pd.read_csv(file, dtype=str, keep_default_na=False, encoding="latin-1")

    # Verify that the necessary 'raw' and 'clean' categories exist
    for col in ("raw", "clean"):
        if col not in df.columns:
            df[col] = ""

    # Check if the specific 'Not Applicable' pair already exists in the data
    na_exists = ((df["raw"] == "Not Applicable") & (df["clean"] == "Not Applicable")).any()

    # Append the row only if it is missing from the current records
    if not na_exists:
        df.loc[len(df)] = {"raw": "Not Applicable", "clean": "Not Applicable"}

        # Overwrite the file with the updated information in a standardized format
        df.to_csv(file, index=False, encoding="utf-8")
        print(f"Successfully updated: {file.name}")


In [ ]:
# 5. Now apply the lookups (Single-Response Columns)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# At this point, lookups are already created in lookup_dir and manually revised.
# df_raw is still unchanged at this point. A new df_single_no_grps is created below.
# Create apply_lookup function using df and column
# Apply the created lookups and print unmatched items

single_cols_no_grps = [
    "resp_id", "country", "city_ph", "city_non_ph", "gender", "age", "educstat", "have_comp_degree", "comp_related_degree", "careerstg", "employertype", "industry", "salary", "how_long_in_salary", "typework", "sitework", "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam", "ai_work", "ai_study", "resp_id_rand"
]
df_raw = pd.read_parquet("parquet_outputs_dir/df_raw.parquet")
df_single_no_grps = df_raw[single_cols_no_grps].copy()

# Export before lookup
df_single_no_grps.to_csv(csv_outputs_dir/"df_single_no_grps_before_lookup.csv", index=False)  

# Re-run the single_column_non_numerics definition
single_columns_non_numeric = [
    col for col in df_single_no_grps.columns
    if col not in ["resp_id", "age", "resp_id_rand", "satisfaction"]
]


In [ ]:
# 5a. Apply lookup function to single - response columns
# Create check unique values in df_single_no_grps and apply lookup
# → apply_lookup function → convert df_single_no_grps.csv using lookup
def apply_lookup(df, column):
    lookup_file = lookup_dir / f"{column}_lookup.csv"
    if not lookup_file.exists():
        print(f"{column} lookup file not found.")
        return df

    map_df = pd.read_csv(lookup_file, encoding="windows-1252", quotechar='"', quoting=csv.QUOTE_MINIMAL, dtype=str, keep_default_na=False)
    map_df["raw"] = map_df["raw"].fillna("")
    map_df["clean"] = map_df["clean"].fillna("")

    list_raw = list(df[column].unique())
    list_lookup = list(map_df["raw"].unique())

    unmatched = [x for x in list_raw if x not in list_lookup]
    if not unmatched:
        print (f"All items in {column} are matched.")
    else:
        print(f"Unmatched items in {column}:")
        for item in unmatched:
            print(repr(item))  # official string representation

    mapping = dict(zip(map_df["raw"], map_df["clean"]))

    df[column] = df[column].map(mapping).fillna("Not Applicable")
    return df


for col in single_columns_non_numeric:
    df_single_no_grps= apply_lookup(df_single_no_grps, col)
    print("~"*20)

# Logging...
import datetime
print('Run date: ', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
# Update the df_single_no_grps csv file
df_single_no_grps.to_csv(csv_outputs_dir/ "df_single_no_grps_after_lookup.csv", index = False)

All items in country are matched.
~~~~~~~~~~~~~~~~~~~~
All items in city_ph are matched.
~~~~~~~~~~~~~~~~~~~~
All items in city_non_ph are matched.
~~~~~~~~~~~~~~~~~~~~
All items in gender are matched.
~~~~~~~~~~~~~~~~~~~~
All items in educstat are matched.
~~~~~~~~~~~~~~~~~~~~
All items in have_comp_degree are matched.
~~~~~~~~~~~~~~~~~~~~
All items in comp_related_degree are matched.
~~~~~~~~~~~~~~~~~~~~
All items in careerstg are matched.
~~~~~~~~~~~~~~~~~~~~
All items in employertype are matched.
~~~~~~~~~~~~~~~~~~~~
All items in industry are matched.
~~~~~~~~~~~~~~~~~~~~
All items in salary are matched.
~~~~~~~~~~~~~~~~~~~~
All items in how_long_in_salary are matched.
~~~~~~~~~~~~~~~~~~~~
All items in typework are matched.
~~~~~~~~~~~~~~~~~~~~
All items in sitework are matched.
~~~~~~~~~~~~~~~~~~~~
All items in shift_schedule are matched.
~~~~~~~~~~~~~~~~~~~~
All items in datarole are matched.
~~~~~~~~~~~~~~~~~~~~
All items in side_gig are matched.
~~~~~~~~~~~~~~~~~~~~
All items i

# Stop here.  Inspect unmatched items.  Manually edit lookup files if needed.
- No need to run lookup again - second pre-cleaning.
- At this point, df_single_no_grps values are already updated, so cannot re-run item 5. Lookups. 
- If you need to run the subset:
    - Just edit the lookups manually in VSCode, not Excel.
    - Or define the subset, noting that artificial unmatched items will appear.

In [66]:
# 5b. Inspect the updated df_single_no_grps unique values. (Optional)
# During pre-cleaning - this was only to check if unforeseen unmatched values still appeared.
def explore_categorical_cols(col, df):
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')
    print(f'Exploration of {col} column')
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')
     
    # Normal categorical branch
    unique_count = df[col].nunique()
    
    # Skip high-cardinality columns
    if unique_count > 100:
        print(f"Skipping {col} (unique values = {unique_count}, exceeds threshold of 100).")
        print()
        return
   
    print(f'Description of {col} ~~~~~~~~~~')
    print(df[col].describe())
    print()
    print(f'Value counts of {col} ~~~~~~~~~~')
    print(df[col].value_counts(dropna=False))
    print()

single_cols_no_grps
# Loop through categorical columns
for col in single_cols_no_grps:
    explore_categorical_cols(col, df_single_no_grps)

print ("At this point, all unique items should be clean.")
print ("Still no age_grp because we need exact age for the duplication check.")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Exploration of resp_id column
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Skipping resp_id (unique values = 1732, exceeds threshold of 100).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Exploration of country column
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Description of country ~~~~~~~~~~
count            1732
unique             30
top       Philippines
freq             1679
Name: country, dtype: object

Value counts of country ~~~~~~~~~~
country
Philippines             1679
Nigeria                    7
Egypt                      6
United Arab Emirates       4
United Kingdom             3
Canada                     3
Saudi Arabia               3
Uganda                     2
United States              2
Nepal                      2
Peru                       2
Cambodia                   1
Cameroon                   1
Norway                     1
Jordan                     1
Uae                        1
Pakistan         

# Check similarity - single response columns

In [62]:
# 5c. Check similarity - preliminary duplicate checks among single-response columns first
# Set up the cleaned single‑response dataframe that we will compare
df_sim_raw = df_single_no_grps.copy()

# List the columns we will use to score similarity
compare_cols = [
    "country", "city_ph", "city_non_ph", "gender", "age", "educstat",
    "have_comp_degree", "comp_related_degree", "careerstg", "employertype",
    "industry", "salary", "how_long_in_salary", "typework", "sitework",
    "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam",
    "ai_work", "ai_study"
]

# Initialize list to hold the similarity results for each respondent
results = []

# Loop through each respondent one at a time
for idx, row in df_sim_raw.iterrows():

    # Create a row‑vs‑all‑rows comparison so we can see who looks similar
    # Compare is the boolean dataframe that contains results True/False per column
    compare = df_sim_raw[compare_cols] == row[compare_cols]

    # Count how many fields match for each respondent
    match_counts = compare.sum(axis=1)

    # Remove the self‑match so it doesn't interfere with scoring
    match_counts.loc[idx] = -1

    # Get the highest match score for this respondent
    max_score = match_counts.max()

    # Grab all respondents who hit that same top score
    best_idxs = match_counts[match_counts == max_score].index.tolist()

    # Save the result so we can review or export later
    results.append({
        "resp_id": row["resp_id"],
        "best_match_resp_ids": df_sim_raw.loc[best_idxs, "resp_id"].tolist(),
        "best_score": int(max_score),
        "total": len(compare_cols),
        "similarity": max_score / len(compare_cols)
    })

# Turn the collected results into a dataframe
df_sim = pd.DataFrame(results)
df_sim


,resp_id,best_match_resp_ids,best_score,total,similarity
0,1,[1373],15,22,0.681818
1,2,"[232, 420, 1413, 1445]",14,22,0.636364
2,3,[402],18,22,0.818182
3,4,[222],15,22,0.681818
4,5,"[123, 125, 163, 385, 454, 1168]",14,22,0.636364
...,...,...,...,...,...
1727,1728,[1727],22,22,1.000000
1728,1729,[977],11,22,0.500000
1729,1730,[951],20,22,0.909091
1730,1731,"[506, 955, 1451, 1674]",21,22,0.954545


In [27]:
# 5d. Inspect the highest scores
df_sim_sorted = df_sim.sort_values(by = 'similarity', ascending = False)
print(df_sim_sorted[['resp_id', 'best_match_resp_ids', 'similarity']].head(50))
df_sim_sorted.to_csv(csv_outputs_dir/ "df_sim_sorted.csv", index = False)

     resp_id                    best_match_resp_ids  similarity
1726    1727                                 [1728]    1.000000
356      357                                  [101]    1.000000
470      471                                  [502]    1.000000
1573    1574                                 [1099]    1.000000
1627    1628                                 [1123]    1.000000
501      502                                  [471]    1.000000
342      343                                 [1382]    1.000000
576      577                                  [337]    1.000000
336      337                                  [577]    1.000000
591      592                                 [1120]    1.000000
704      705                                 [1081]    1.000000
305      306                                 [1031]    1.000000
780      781                                  [202]    1.000000
838      839                                 [1285]    1.000000
844      845                            

In [67]:
# 5e. Filter respondents most likely to be duplicates 
# Score == 1 & only one best_match_resp_ids using apply(len) == 1)
df_sim_sorted = df_sim.sort_values(by = 'similarity', ascending = False)
df_likely_dups = df_sim_sorted[
    (df_sim_sorted['best_match_resp_ids'].apply(len) == 1) &
    (df_sim_sorted['similarity'] == 1)
]

print(df_likely_dups.head(50))

# Export the df_likely_dups to csv outputs
df_likely_dups.to_csv(csv_outputs_dir/"df_likely_dups.csv", index = False)

      resp_id best_match_resp_ids  best_score  total  similarity
1726     1727              [1728]          22     22         1.0
356       357               [101]          22     22         1.0
470       471               [502]          22     22         1.0
1573     1574              [1099]          22     22         1.0
1627     1628              [1123]          22     22         1.0
501       502               [471]          22     22         1.0
342       343              [1382]          22     22         1.0
576       577               [337]          22     22         1.0
336       337               [577]          22     22         1.0
591       592              [1120]          22     22         1.0
704       705              [1081]          22     22         1.0
305       306              [1031]          22     22         1.0
780       781               [202]          22     22         1.0
838       839              [1285]          22     22         1.0
844       845            

In [ ]:
# 5f. Create df possible duplicates based on the single response columns.

df_possible_duplicates = pd.merge(df_single_no_grps, df_likely_dups, how = "inner", on = "resp_id")
df_possible_duplicates.to_csv(csv_outputs_dir/ "df_possible_duplicates.csv", index = False)
df_possible_duplicates

# 6. Create groups for age and salary

In [69]:
# 6a. Create age_grp in df_single_no_grps
# Initialize the df_single_with_grps

df_single_no_grps = pd.read_csv("csv_outputs_dir/df_single_no_grps_after_lookup.csv")
df_single_with_grps = df_single_no_grps.copy()

# Bin the 'age' variable into 'age_grp'
bins = [-np.inf, 19, 25, 30, 35, 40, 45, 50, 55, np.inf]
labels = ["<19", "20 to 24", "25 to 29", "30 to 34", "35 to 39", "40 to 44", "45 to 49", "50 to 54", "55+"]
df_single_with_grps['age_grp'] = pd.cut(df_single_no_grps['age'], bins=bins, labels=labels, right=False, include_lowest=True)
print("age_grp column added to df_single_with_grps.")


age_grp column added to df_single_with_grps.


In [ ]:
# Check df_single_with_grps
df_single_with_grps

In [74]:
# 6b. Create column salary_broader for plotly splits

# Define broader salary bands using match-case
def categorize_salary(salary):
    match salary:
        case x if x in ['15k and below', '15k+ to 25k', '25k+ to 35k']:
            return "35k and less"
        case x if x in ['35k+ to 45k', '45k+ to 55k', '55k+ to 65k', '65k+ to 75k']:
            return "35k+ to 75k"
        case x if x in ['75k+ to 85k', '85k+ to 95k', '95k+ to 100k']:
            return "75k+ to 100k"
        case x if x in ['100k+ to 125k', '125k+ to 250k', '250k and above']:
            return "100k and above"   
        case x if x == 'Not Applicable':
            return "Not Applicable"
        case _:
            return "Unknown"
df_single_with_grps['salary_broader'] = df_single_with_grps['salary'].apply(categorize_salary)
df_single_with_grps.to_csv("final_outputs_dir/df_single_with_grps.csv", index = False )
# df_single_with_grps.to_csv("csv_outputs_dir/df_single_with_grps.csv", index = False)
print("salary_broader column added to df_single_with_grps. ")


salary_broader column added to df_single_with_grps. 


In [ ]:
# Check df_single_with_grps
df_single_with_grps

In [ ]:
# Check if any salary_broader == 'Unknown'
unknown_salary_check = df_single_with_grps[df_single_with_grps['salary_broader'] == "Unknown"][['resp_id', 'salary', 'salary_broader']]
unknown_salary_check

,resp_id,salary,salary_broader


In [76]:
# 6c. Export df_single_with_grps
# To csv
df_single_no_grps.to_csv( csv_outputs_dir/ "df_single_no_grps.csv", index = False)
df_single_with_grps.to_csv( csv_outputs_dir/ "df_single_with_grps.csv", index = False)
df_single_with_grps.to_csv( final_outputs_dir/ "df_single_with_grps.csv", index = False)
# To parquet
df_single_with_grps.to_parquet(parquet_outputs_dir/"df_single_with_grps.parquet", index = False)

# 7. Handle multi-response columns
- Define multi_response_cols list
- Create explode_and_lookup function

In [6]:
# 0. Imports & Config
# ~~~~~~~~~~~~~~~~~~~~~~~~~
import pandas as pd
import unicodedata
from pathlib import Path
import numpy as np
import csv # so comma delimited values are not split
import datetime

In [2]:
# 0. Create Directories
# Store all lookup files for single-response columns and exploded multi-response csv files
lookup_dir = Path("lookup_dir")
lookup_dir.mkdir(exist_ok=True)

# Store all parquet files
parquet_outputs_dir = Path("parquet_outputs_dir")
parquet_outputs_dir.mkdir(exist_ok=True)

# Store all the outputs as csv so cleaning can be split into modules
csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)

# Copy of all the final outputs for data model
final_outputs_dir = Path("final_outputs_dir")
final_outputs_dir.mkdir(exist_ok=True)

# Store preliminary unique values
unique_dir = Path("unique_dir")
unique_dir.mkdir(exist_ok=True)

# Store all csv and map files using country and city columns
location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)

In [4]:
# 7. Define and filter multi-response columns

# Import df_raw if starting this section again
df_raw = pd.read_parquet(parquet_outputs_dir/"df_raw.parquet")
df_raw

# Define multi-response columns
multi_response_cols = ["digitools", "successmethod", "restofrole", "ingestion", "transform", "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools", "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds", "otherfb", "attended_inperson", "attended_online"]

# filter using multi-response columns
df_raw_multi = df_raw[['resp_id'] + multi_response_cols].copy()
df_raw_multi.columns

Index(['resp_id', 'digitools', 'successmethod', 'restofrole', 'ingestion',
       'transform', 'storage', 'warehs', 'orchest', 'busint', 'cloudplat',
       'generaltools', 'whatused', 'ai_tools', 'hardware', 'dep_init',
       'volunteer', 'spneeds', 'otherfb', 'attended_inperson',
       'attended_online'],
      dtype='object')

In [ ]:
df_raw

In [7]:
# 7a1. Remove comma in MS Fabric under ingestion
# print (df_raw_multi['ingestion'].unique())

# Replace the MS Fabric item
df_raw_multi['ingestion'] = df_raw_multi['ingestion'].str.replace("MS Fabric (Eventstreams, Dataflow, COPY, etc.)", "MS Fabric (Evenstreams Dataflow COPY)")

# Inspect again
# Filter the column first, then get unique values
ms_fabric_ingestion = df_raw_multi[df_raw_multi['ingestion'].str.contains("MS Fabric")]['ingestion']
print(ms_fabric_ingestion.unique())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

['MS Excel, MS Fabric (Evenstreams Dataflow COPY), MS Power Platform, Power BI'
 'Azure Data Factory, MS Fabric (Evenstreams Dataflow COPY), MSSQL replication, Power BI, SSIS'
 'Alteryx, Amazon Kinesis, Apache Kafka, AWS Glue, AWS Lambda, Azure Data Factory, Azure Databricks, Azure Synapse Analytics, Databricks, Dataverse, Fivetran, Google Sheets, Google Cloud Composer, Google Cloud Dataflow, Hadoop, Knime, MS Excel, MS Fabric (Evenstreams Dataflow COPY), MS Forms, MS Power Platform, MSSQL replication, Oracle, Power BI, Python scripts, Salesforce, Snowflake, Spark, SQL, SSIS, Tableau Prep, Talend'
 'Azure Data Factory, Azure Databricks, Azure Synapse Analytics, Databricks, MS Fabric (Evenstreams Dataflow COPY), Power BI, Python scripts, Snowflake, Spark, SQL'
 'AWS Glue, Azure Databricks, MS Fabric (Evenstreams Dataflow COPY), MS Power Platform, Power BI, SQL, Tableau Prep'
 'Google Sheets, MS Excel, MS Fabric (Evenstreams Dataflow COPY), MS Forms, MS Power Platform, Tableau Prep'
 'AW

In [8]:
# 7a2. Remove comma in MS Fabric under orchestration
# print (df_raw_multi['orchest'].unique())

# Replace the MS Fabric item under orchestration
df_raw_multi['orchest'] = df_raw_multi['orchest'].str.replace("MS Fabric (Pipelines, etc.)", "MS Fabric (Pipelines)")

# Inspect again
ms_fabric_orchestration = df_raw_multi[df_raw_multi['orchest'].str.contains("MS Fabric")]['orchest']
print(ms_fabric_orchestration.unique())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


['MS Fabric (Pipelines), SQL Server Agent'
 'Airflow, Astronomer, AWS Step Functions, Azure Data Factory, Cron, Databricks Workflows, GCP Cloud Workflows, Kubeflow, MS Fabric (Pipelines), MS Power Automate, SQL Server Agent, Windows Task Scheduler'
 'Airflow, MS Fabric (Pipelines), MS Power Automate'
 'AWS Step Functions, MS Fabric (Pipelines)' 'MS Fabric (Pipelines)'
 'Azure Data Factory, Databricks Workflows, MS Fabric (Pipelines)'
 'Databricks Workflows, MS Fabric (Pipelines), MS Power Automate'
 'MS Fabric (Pipelines), MS Power Automate'
 'Databricks Workflows, Kubeflow, MS Fabric (Pipelines)'
 'Azure Data Factory, MS Fabric (Pipelines), Orkes'
 'Azure Data Factory, MS Fabric (Pipelines)'
 'MS Fabric (Pipelines), MS Power Automate, SQL Server Agent'
 'Azure Data Factory, MS Fabric (Pipelines), MS Power Automate'
 'MS Fabric (Pipelines), MS Power Automate, Windows Task Scheduler, None of the above'
 'Azure Data Factory, MS Fabric (Pipelines), MS Power Automate, Sematic, SQL Server A

In [9]:
# 7a3. Remove comma in MS Fabric under transformation
# print (df_raw_multi['transform'].unique())

# Replace the MS Fabric item under transformation
df_raw_multi['transform'] = df_raw_multi['transform'].str.replace("MS Fabric (dbt-fabric, Dataflow, etc.)", "MS Fabric (dbt-fabric Dataflow)")

# Inspect again
ms_fabric_transformation = df_raw_multi[df_raw_multi['transform'].str.contains("MS Fabric")]['transform']
print(ms_fabric_transformation.unique())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


['MS Fabric (dbt-fabric Dataflow), MS Power BI Dataflows, SQL-Based Platforms'
 'Alteryx, AWS Athena, AWS Glue, Azure Data Factory, Dataverse, Google BigQuery, Google Sheets, Informatica, Knime, MS Excel, MS Fabric (dbt-fabric Dataflow), MS Power Apps, Pandas, Polars, Postgresql, MS Power Automate, MS Power Query, MS Power BI Dataflows, Pyspark, Python, R, Spark, SQL-Based Platforms, SQL/Full SQL(BigQuery), SQL/PL/SQL(Oracle), SQL/Postgresql, SQL/Snowflake, SQL/SparkSQL(Databricks), SQL/T-SQL, SSIS, Tableau Prep, XSLT'
 'MS Excel, MS Fabric (dbt-fabric Dataflow), MS Power Apps, MS Power Automate, MS Power Query, MS Power BI Dataflows'
 'Azure Data Factory, MS Fabric (dbt-fabric Dataflow), Pyspark, Spark, SQL/Snowflake, SQL/SparkSQL(Databricks)'
 'Azure Data Factory, MS Fabric (dbt-fabric Dataflow), Python, SQL-Based Platforms'
 'Google Sheets, Informatica, MS Excel, MS Fabric (dbt-fabric Dataflow), MS Power Apps, R'
 'Azure Data Factory, MS Excel, MS Fabric (dbt-fabric Dataflow), SQL/P

In [58]:
# 7b. Special cleaning for attended inperson and online - clean and standardize None responses
# Using fuzzy match
cols_attended = ['attended_inperson', 'attended_online']

for col in cols_attended:

    # Fuzzy match patterns
    mask_fuzzy = df_raw_multi[col].str.lower().str.contains(
        r'non|none|nope|n/a',
        na=False
    )

    # Exact none-like values
    mask_exact = df_raw_multi[col].str.strip().str.lower().isin([
        'nan',
        '/a',
        'n/a',
        'nope',
        'no e',
        'npne',
        'nine',
        'nin',
        'n n'
    ])

    # True NaN values
    mask_nan = df_raw_multi[col].isna()

    # Combine all masks
    mask_combined = mask_fuzzy | mask_exact | mask_nan

    # Replace with standard value
    df_raw_multi.loc[mask_combined, col] = 'None'

    # Final normalization
    df_raw_multi[col] = df_raw_multi[col].str.strip().str.upper()

    # Print unique values
    unique_list_attended= sorted(set(df_raw_multi[col].to_list()))
    print(f"Unique values in {col}")
    print("~"*40)
    print(unique_list_attended)
    print("~"*40, "\n\n")

# Export df_raw_multi to csv
df_raw_multi.to_csv(csv_outputs_dir/"df_raw_multi.csv", index = False)  
df_raw_multi.to_parquet(parquet_outputs_dir/"df_raw_multi.parquet", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Unique values in attended_inperson
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
['"NEXT-GEN JAVA: AI, CLOUD, & PERFORMANCE" SEMINAR', '2025 PHILIPPINE JUNIOR DATA SCIENCE CHALLENGE', '2ND MEET UP IN CBTL', 'AIBP CONVENTION, DATA STREAMS WORKSHOPS', 'ATTENDED A SEMINAR ORGANIZED BY AI PILIPINAS', 'AWS CLOUD DAY', 'AWS COMMUNITY DAY', 'AWS INNOVATION CUP', 'AWS STUDENT COMMUNITY NIGHT', 'BIBE', 'BIG DATA EUROPE', 'BPI DATA WAVE, DICT PSC', 'BPI WAVE HACKATHON', 'CTRL ALT RUN', 'DATA ENGINEERING BULACAN MEETUP', 'DATA FUTURE PHILIPPINES 2025', 'DAXDAKAN', 'DAXDAKAN LIVE', 'DEEPLEARNING.AI HACKATHON', 'DEP DECEMBER CONFERENCE', 'DEP SOUTH MEETUP', 'DEVCON', 'DEVCON MORE ABOUT ON AI', 'DICT STARTUP CONTEST', 'DSA ONLINE TRAINING', 'EHEADS', 'EHEADS, PYCON APAC 2025, AI PIE MANILA', 'EHEADS20', 'FEU TECH CS EXPO', 'FREE WEBINARS, SHORT COURSES ONLINE', 'FTW FOUNDATION DATA ANALYTICS PROGRAM', 'GDG GOOGLE DEV SEMINARS AND WORKSHOPS', 'GOOGLE DEV FEST', 'GOOGLE DEVFEST 2025', 'GOOGLE DEVFEST DAVAO

In [16]:
# 7c. Transform and normalize string values, explode and normalize functions for multi-response columns

# Transform string values (nested function inside explode and normalize)
def normalize_text(val):
    if pd.isna(val):
        return ""
    val = unicodedata.normalize("NFKC", str(val))
    val = val.strip()
    val = " ".join(val.split())
    return val


def explode_and_normalize(df, column):
    # Include resp_id and multi-response columns
    exploded = (
        df[["resp_id", column]]
        .assign(**{column: df[column].str.split(",")})
        .explode(column)
    )
    exploded[column] = exploded[column].apply(normalize_text)

    # Remove duplicates (resp_id, value) pairs
    exploded = exploded.drop_duplicates(subset=["resp_id", column], keep="first")

    exploded.to_parquet(parquet_outputs_dir / f"{column}_exploded.parquet", index = False)
    exploded.to_csv(csv_outputs_dir/f"{column}_exploded.csv", index = False)
    return exploded

print("Functions normalize_text and explode and normalize created")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Functions normalize_text and explode and normalize created
Date and Time:  2026-03-10 09:36:21


In [17]:
# 7d. Apply explode_and_normalize function → csv, parquet

# Now that the multi-response columns are cleaned, initialize df_multi dataframe from df_raw_multi

df_raw_multi=pd.read_csv(csv_outputs_dir/"df_raw_multi.csv")
df_multi = df_raw_multi[["resp_id"] + multi_response_cols].copy()

# Initialize the df_multi csv file
df_multi.to_csv(csv_outputs_dir / "df_multi.csv", index=False)

# Explode each multi-response column
for col in multi_response_cols:
    # Explode and lookup for this column
    df_exploded = explode_and_normalize(df_multi, col)

    # Export exploded dataframe as {col}_exploded.csv and parquet
    df_exploded.to_csv(csv_outputs_dir / f"{col}_exploded.csv", index=False)
    df_exploded.to_parquet(parquet_outputs_dir / f"{col}_exploded.parquet", index=False)

print("Multi-response columns exploded → csv, parquet.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Multi-response columns exploded → csv, parquet.
Date and Time:  2026-03-10 09:36:34


In [18]:
# 7e. Special cleaning for datarole and restofrole 
# Check datarole duplicates by merging datarole and restofrole
# If datarole and restofrole are the same, then they are duplicates
# Create helper column
df_restofrole_exploded = pd.read_csv(csv_outputs_dir/"restofrole_exploded.csv")
df_single_with_grps = pd.read_csv(csv_outputs_dir/"df_single_with_grps.csv")
df_datarole = df_single_with_grps[['resp_id', 'datarole']]
df_role_merged = df_datarole.merge(df_restofrole_exploded, on = 'resp_id', how = 'inner')
df_role_merged['duplicated'] = (
    df_role_merged['datarole'].str.strip().str.lower()
    == df_role_merged['restofrole'].str.strip().str.lower()
)


# Drop duplicated rows in df_role_merged
df_role_merged = df_role_merged[df_role_merged['duplicated'] == False]

# Drop the helper column
df_role_merged = df_role_merged.drop(columns=['duplicated'])
df_role_merged.to_csv(csv_outputs_dir/"df_role_merged.csv", index = False)


# Create an updated restofrole exploded csv with a longer name and shorter name - for general cleaning later
df_restofrole_before_general_cleaning = df_role_merged[['resp_id', 'restofrole']].copy()
df_restofrole_before_general_cleaning.to_csv(csv_outputs_dir/"restofrole_exploded_before_general_cleaning.csv", index = False)
df_restofrole_before_general_cleaning.to_csv(csv_outputs_dir/"restofrole_exploded.csv", index = False)

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Special cleaning of restofrole done.")

Date and Time:  2026-03-10 09:36:56
Special cleaning of restofrole done.


In [59]:
# 7f. Second round of clean up multi-response columns after being exploded, especially the None variants

multi_response_cols = ["digitools", "successmethod", "restofrole", "ingestion", "transform", "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools", "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds", "otherfb", "attended_inperson", "attended_online"]

none_variants = [
    "None", "None ", "Non", "non", "non ", "npne", "n ne", "npn", "nine", "nin", "nond", "nonf"]

for col in multi_response_cols:
    path = csv_outputs_dir / f"{col}_exploded.csv"
    df_exp_cln = pd.read_csv(path)

    # Convert to string, strip spaces
    df_exp_cln[col] = df_exp_cln[col].astype("string").str.strip()

    # First, standardize blank-like values replace with pd.NA
    df_exp_cln[col] = df_exp_cln[col].replace(
        ["", "nan", "NaN"], pd.NA
    )

    # Then, normalize None-like strings to "None"
    df_exp_cln[col] = df_exp_cln[col].replace(none_variants, "None")

    # Drop rows where the exploded value is NA
    df_exp_cln = df_exp_cln.dropna(subset=[col])

    # Drop blanks
    df_exp_cln = df_exp_cln[df_exp_cln[col] != ''    ]

    # Save cleaned version
    df_exp_cln.to_csv(csv_outputs_dir / f"{col}_exploded_cleaned.csv", index=False)
    df_exp_cln.to_parquet(parquet_outputs_dir / f"{col}_exploded_cleaned.parquet", index=False)

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Second round of cleaning of multi-response columns done.")

Date and Time:  2026-03-10 12:18:21
Second round of cleaning of multi-response columns done.


In [60]:
# 7g. Summary of cleaned multi-response columns
# Recall the multi-response columns definition
multi_response_cols = [
    "digitools", "successmethod", "restofrole", "ingestion", "transform",
    "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools",
    "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds",
    "otherfb", "attended_inperson", "attended_online",
]

# Initialize the summary_frames list for pd.concat later
summary_frames = []

# Build summary of cleaned multi-response columns
def build_summary_multi(col):
    for col in multi_response_cols:
        # Build summary for this column
        path = csv_outputs_dir / f"{col}_exploded_cleaned.csv"
        df_exploded = pd.read_csv(path)

        # Remove duplicates on (resp_id, col) pair, retain first
        df_exploded = df_exploded.drop_duplicates(subset=["resp_id", col], keep="first")

        value_counts = df_exploded[col].value_counts(dropna=False).reset_index()
        value_counts.columns = ["value", "count"]
        value_counts.insert(0, "column", col)
        summary_frames.append(value_counts)

        # Divider row
        divider = pd.DataFrame([{"column": "~" * 50, "value": "", "count": ""}])
        summary_frames.append(divider)

# Call build summary        
build_summary_multi(multi_response_cols)

# Combine all summaries and export
summary_combined = pd.concat(summary_frames, ignore_index=True)
summary_combined.to_csv(csv_outputs_dir / "multi_response_summary.csv", index=False)

print("Summary of exploded cleaned tables created.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Summary of exploded cleaned tables created.
Date and Time:  2026-03-10 12:18:26


In [61]:
#7i. Check similarity of df_possible_duplicates including multi-response columns
# Load dfs -> possible duplicates (single), merge with multi-response cols 
# -> split items list from string → "" blanks →  strip spaces  → sort → ", ".join to string
# Exploded files are unchanged.

df_possible_duplicates = pd.read_csv(csv_outputs_dir/"df_possible_duplicates.csv")
df_multi = pd.read_csv(csv_outputs_dir/"df_multi.csv")
df_likely_dups = pd.read_csv(csv_outputs_dir/"df_likely_dups.csv")

multi_response_cols = [
    "digitools", "successmethod", "restofrole", "ingestion", "transform",
    "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools",
    "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds",
    "otherfb", "attended_inperson", "attended_online"
]

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Merge df_possible_duplicates with df_multi FIRST (inner join)
# This keeps only respondents present in both tables
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

df_merged_dups = df_possible_duplicates.merge(
    df_multi[["resp_id"] + multi_response_cols],
    on="resp_id",
    how="inner"
)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Standardization function for multi-response columns
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def standardize_multi_response(s):
    if pd.isna(s) or s.strip() == "":
        return ""
    items = [x.strip() for x in s.split(",")]
    items = sorted([x for x in items if x != ""])
    return ", ".join(items)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Apply standardization ONLY to the merged subset
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

for col in multi_response_cols:
    df_merged_dups[col] = df_merged_dups[col].apply(standardize_multi_response)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Build explicit duplicate pairs from df_likely_dups
# Each row has resp_id and a list like [577]
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

resp_b = df_likely_dups["best_match_resp_ids"].str.strip("[] ").astype(int)

df_pairs = pd.DataFrame({
    "resp_id_1": df_likely_dups["resp_id"].combine(resp_b, min),
    "resp_id_2": df_likely_dups["resp_id"].combine(resp_b, max),
}).drop_duplicates()


# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Compare multi-response columns for each pair
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

results = []

for _, row in df_pairs.iterrows():
    id1 = row["resp_id_1"]
    id2 = row["resp_id_2"]

    row_a = df_merged_dups[df_merged_dups["resp_id"] == id1].iloc[0]
    row_b = df_merged_dups[df_merged_dups["resp_id"] == id2].iloc[0]

    comparison = {
        "resp_id_1": id1,
        "resp_id_2": id2,
    }

    for col in multi_response_cols:
        comparison[col] = (row_a[col] == row_b[col])

    results.append(comparison)

# Convert results list to dataframe
df_multi_compare = pd.DataFrame(results)
df_multi_compare['count_true'] =df_multi_compare[multi_response_cols].sum(axis=1)
df_multi_compare['count_false'] = df_multi_compare.shape[1] - df_multi_compare['count_true']

df_multi_compare.to_csv(csv_outputs_dir/"df_multi_compare.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Duplicate check using multi-response columns done.")


Date and Time:  2026-03-10 12:18:58
Duplicate check using multi-response columns done.


In [64]:
# Full duplicate test
print(df_multi_compare[['resp_id_1', 'resp_id_2', 'count_true', 'count_false']])

print("~"*75)
print("If count_false > 0, then resp1 and resp2 are not duplicates.")
print("~"*75, "\n")

# Check for possible duplicates
if (df_multi_compare['count_false'] == 0).any():
    print("Possible duplicates found.")
else:
    print("No possible duplicates.")

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*75, "\n")


# Second run - multi-response columns are different. Nothing to delete.

    resp_id_1  resp_id_2  count_true  count_false
0        1727       1728          11           12
1         101        357          12           11
2         471        502          11           12
3        1099       1574          11           12
4        1123       1628          13           10
5         343       1382          10           13
6         337        577          11           12
7         592       1120          12           11
8         705       1081          11           12
9         306       1031          17            6
10        202        781          10           13
11        839       1285          10           13
12        845       1211          10           13
13        239       1418          11           12
14        961       1419          12           11
15        218       1315          12           11
16       1534       1624          13           10
17        195       1189          12           11
18        198       1337          13           10


# 8.  Lineage - location dfs

- df_loc_new <- df_single_with_groups
- df_ph_grp, df_non_ph_grp <- df_loc_new 
- overseas <- df_ph_grp 
- df_ph_geo_clean <- df_ph_geo
- df_non_ph_geo_clean <- df_non_ph_geo
- df_all_geo_clean <- df_ph_geo_clean, df_non_ph_geo_clean

In [34]:
# 8. Load df_loc_new and save
# ~~~~~~~~~~~~~~~~~~~~~~~~~
df_single_with_grps = pd.read_csv(csv_outputs_dir/ "df_single_with_grps.csv")
df_loc_new = df_single_with_grps[["resp_id", "country", "city_ph", "city_non_ph"]].drop_duplicates()

location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)
df_loc_new.to_csv(location_dir/ "df_loc_new.csv", index = False)

df_loc_new.info()
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1732 entries, 0 to 1731
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resp_id      1732 non-null   int64 
 1   country      1732 non-null   object
 2   city_ph      1732 non-null   object
 3   city_non_ph  1732 non-null   object
dtypes: int64(1), object(3)
memory usage: 54.2+ KB
Date and Time:  2026-03-10 10:35:30


In [40]:
# 8a. Check non-ph cities
# If Prefer Not To Say or Not Applicable change to capital city later using lookup
not_in_ph = df_loc_new[df_loc_new['country'] != "Philippines"]
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
not_in_ph



Date and Time:  2026-03-10 10:44:18
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,resp_id,country,city_ph,city_non_ph
5,6,United Kingdom,Not Applicable,Newcastle
20,21,United Kingdom,Not Applicable,London
157,158,Norway,Not Applicable,Oslo
183,184,Cameroon,Not Applicable,Douala
232,233,Cambodia,Not Applicable,Phnom Penh
304,305,Thailand,Not Applicable,Bangkok
319,320,Pakistan,Diplo,Diplo
396,397,United Arab Emirates,Not Applicable,UAE
543,544,Canada,Not Applicable,Prefer Not To Say
741,742,Canada,Caloocan,Not Applicable


In [70]:
# 8c. Check ph cities
df_ph_grp = df_loc_new[df_loc_new['country'] == "Philippines"].groupby(['country','city_ph', 'city_non_ph'])['resp_id'].count().reset_index()
df_ph_grp.rename(columns = {"resp_id": "count_resp"}, inplace = True)
df_ph_grp.to_csv(location_dir/"df_ph_grp.csv", index=False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
df_ph_grp

Date and Time:  2026-03-10 12:51:26
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_ph,city_non_ph,count_resp
0,Philippines,Aklan,Not Applicable,1
1,Philippines,Albay,Not Applicable,10
2,Philippines,Aleosan,Not Applicable,1
3,Philippines,Angeles,Not Applicable,1
4,Philippines,Angono,Not Applicable,3
...,...,...,...,...
184,Philippines,Virac,Not Applicable,1
185,Philippines,Western Samar,Not Applicable,1
186,Philippines,Zambales,Not Applicable,5
187,Philippines,Zambales,Prefer Not To Say,1


In [71]:
# 8d. Cannot force city_non_ph to be "Not Applicable" if country = Philippines
# Lots of 'Prefer Not To Say', likely working overseas

overseas = df_ph_grp[df_ph_grp['city_non_ph'] == "Prefer Not To Say"]
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
overseas

Date and Time:  2026-03-10 12:51:34
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_ph,city_non_ph,count_resp
6,Philippines,Antipolo,Prefer Not To Say,1
19,Philippines,Batangas,Prefer Not To Say,1
32,Philippines,Butuan,Prefer Not To Say,1
38,Philippines,Cagayan De Oro,Prefer Not To Say,2
44,Philippines,Caloocan,Prefer Not To Say,2
53,Philippines,Cavite,Prefer Not To Say,1
55,Philippines,Cebu,Prefer Not To Say,1
75,Philippines,Iligan City,Prefer Not To Say,1
105,Philippines,Mandaluyong,Prefer Not To Say,2
127,Philippines,Not Applicable,Prefer Not To Say,1


In [76]:
# 8e. Check city_non_ph if complete for geocoding
df_non_ph_grp = df_loc_new[df_loc_new['country'] != "Philippines"].groupby(['country','city_non_ph', 'city_ph'])['resp_id'].count().reset_index()
df_non_ph_grp.to_csv(location_dir/"df_non_ph_grp_before_lookup.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
df_non_ph_grp

Date and Time:  2026-03-10 13:03:07
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_non_ph,city_ph,resp_id
0,Algeria,Not Applicable,Not Applicable,1
1,Bangladesh,Barishal,Not Applicable,1
2,Cambodia,Phnom Penh,Not Applicable,1
3,Cameroon,Douala,Not Applicable,1
4,Canada,Not Applicable,Caloocan,1
5,Canada,Prefer Not To Say,Not Applicable,1
6,Canada,Regina,Not Applicable,1
7,Congo,Kinshasa,Kinshasa,1
8,Egypt,Alexandria,Alexandria,1
9,Egypt,Cairo,Cairo,1


In [48]:
# 8f. Capital lookup for missing city_non_ph
import pandas as pd
df_non_ph_grp = pd.read_csv(location_dir/"df_non_ph_grp_before_lookup.csv")

# Extract all unique countries
unique_countries = sorted(df_non_ph_grp["country"].dropna().unique())

# Lookup dictionary only for missing cities
capital_lookup = {
    "Algeria": "Algiers",
    "Bangladesh": "Dhaka",
    "Cambodia": "Phnom Penh",
    "Cameroon": "Yaounde",
    "Canada": "Ottawa",
    "Congo": "Kinshasa",
    "Egypt": "Cairo",
    "Ghana": "Accra",
    "India": "New Delhi",
    "Jordan": "Amman",
    "Kenya": "Nairobi",
    "Malaysia": "Kuala Lumpur",
    "Nepal": "Kathmandu",
    "Nigeria": "Abuja",
    "Norway": "Oslo",
    "Pakistan": "Islamabad",
    "Peru": "Lima",
    "Saudi Arabia": "Riyadh",
    "South Africa": "Pretoria",
    "South Korea": "Seoul",
    "Spain": "Madrid",
    "Thailand": "Bangkok",
    "Uae": "Abu Dhabi",
    "Uganda": "Kampala",
    "United Arab Emirates": "Abu Dhabi",
    "United Kingdom": "London",
    "United States": "Washington",
    "Vietnam": "Hanoi",
    "Yemen": "Sanaa",
}

# Find missing countries
missing = [c for c in unique_countries if c not in capital_lookup]

if len(missing) == 0:
    print("All countries are in the lookup dictionary.")
else:
    print ("Some countries are missing from the lookup")
    for c in missing:
        print("-", c)

# Stop here and manually update capital_lookup
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)


All countries are in the lookup dictionary.
Date and Time:  2026-03-10 10:59:04
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [65]:
# 8g. Replace missing or suppressed city_non_ph values with capital
# Geocode non ph cities -> wait 35 mins or less

df_non_ph_grp["city_non_ph"] = df_non_ph_grp.apply(
    lambda row: capital_lookup[row["country"]]
    if row["city_non_ph"] in ["Not Applicable", "Prefer Not To Say"]
    else row["city_non_ph"],
    axis=1
)

df_non_ph_grp.to_csv(location_dir/"df_non_ph_grp_after_lookup.csv", index = False)
df_non_ph_grp.to_csv(location_dir/"df_non_ph_grp.csv", index = False)
print("Missing values replaced with capitals")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
df_non_ph_grp

Missing values replaced with capitals
Date and Time:  2026-03-10 12:22:59
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_non_ph,resp_id
0,Algeria,Algiers,1
1,Bangladesh,Barishal,1
2,Cambodia,Phnom Penh,1
3,Cameroon,Douala,1
4,Canada,Ottawa,1
5,Canada,Ottawa,1
6,Canada,Regina,1
7,Congo,Kinshasa,1
8,Egypt,Alexandria,1
9,Egypt,Cairo,4


# Geocode 1.non ph, 2. ph cities → combine

In [ ]:
# 8h. Geocode non ph cities
# ~~~~~~~~~~~~~~~~~~~~~~~~~
# Import geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize geocoder
geolocator = Nominatim(user_agent="non_ph_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# Extract unique cities for geocoding
unique_cities = df_non_ph_grp["city_non_ph"].dropna().unique()

# Prepare list for results
geo_results = []

for city in unique_cities:
    query = f"{city}"
    location = geocode(query)

    if location:
        geo_results.append(
            {
                "city_non_ph": city,
                "latitude": location.latitude,
                "longitude": location.longitude
            }
        )
    else:
        geo_results.append(
            {
                "city_non_ph": city,
                "latitude": None,
                "longitude": None
            }
        )

# Convert geocoding results to dataframe
df_geo = pd.DataFrame(geo_results)

# Merge coordinates back into main dataframe
df_non_ph_geo = df_non_ph_grp.merge(df_geo, on="city_non_ph", how="left")
df_non_ph_geo = df_non_ph_geo.rename(columns={"resp_id": "count_resp"})

# Export final geocoded file
df_non_ph_geo.to_csv(location_dir/"df_non_ph_geo.csv", index=False)

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Need human check because of geopy mistakes.")
print("~"*50)
df_non_ph_geo


In [77]:
# 8i. Geocode ph cities
# The whole process takes 4-5 mins

# Import geopy
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

df_ph_grp = pd.read_csv(location_dir/"df_ph_grp.csv")

# Initialize geocoder
geolocator = Nominatim(user_agent="ph_city_geocoder")

# Rate limiter: 1 request per second
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# Extract unique PH cities
unique_cities = df_ph_grp["city_ph"].dropna().unique()

# Prepare list for results
geo_results = []

for city in unique_cities:
    query = f"{city}, Philippines"
    location = geocode(query)

    if location:
        geo_results.append(
            {
                "city_ph": city,
                "latitude": location.latitude,
                "longitude": location.longitude,
            }
        )
    else:
        geo_results.append(
            {
                "city_ph": city,
                "latitude": None,
                "longitude": None,
            }
        )

# Convert geocoding results to dataframe
df_geo = pd.DataFrame(geo_results)

# Merge coordinates back into grouped dataframe
df_ph_geo = df_ph_grp.merge(df_geo, on="city_ph", how="left")

# Export final geocoded file
df_ph_geo.to_csv(location_dir / "df_ph_geo.csv", index=False)


print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Need human check because of geopy mistakes.")
print("~"*50)
df_ph_geo


Date and Time:  2026-03-10 13:10:41
Need human check because of geopy mistakes.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_ph,city_non_ph,count_resp,latitude,longitude
0,Philippines,Aklan,Not Applicable,1,11.629789,122.248118
1,Philippines,Albay,Not Applicable,10,13.216667,123.550000
2,Philippines,Aleosan,Not Applicable,1,7.158581,124.577400
3,Philippines,Angeles,Not Applicable,1,15.138935,120.587532
4,Philippines,Angono,Not Applicable,3,14.525848,121.152968
...,...,...,...,...,...,...
184,Philippines,Virac,Not Applicable,1,13.580359,124.231203
185,Philippines,Western Samar,Not Applicable,1,11.833333,125.000000
186,Philippines,Zambales,Not Applicable,5,15.230000,120.120000
187,Philippines,Zambales,Prefer Not To Say,1,15.230000,120.120000


In [80]:
# 8i. Clean and combine ph and non ph dfs

# Create cleaned files for folium
df_ph_geo_clean = df_ph_geo.dropna(subset=["latitude", "longitude"])
df_ph_geo_clean.to_csv(location_dir/ "df_ph_geo_clean.csv", index = False)

df_non_ph_geo_clean = df_non_ph_geo.dropna(subset=["latitude", "longitude"])
df_non_ph_geo_clean.to_csv(location_dir/ "df_non_ph_geo_clean.csv", index = False)

df_ph_geo_clean = pd.read_csv(location_dir/"df_ph_geo_clean.csv")
df_non_ph_geo_clean = pd.read_csv(location_dir/"df_non_ph_geo_clean.csv")

# Enforce identical column order and names
required_cols = [
    "country",
    "city_ph",
    "city_non_ph",
    "count_resp",
    "latitude",
    "longitude"
]

df_ph_geo_clean= df_ph_geo_clean[required_cols]
df_non_ph_geo_clean = df_non_ph_geo_clean[required_cols]

# Concatenate
df_all_geo_clean = pd.concat([df_ph_geo_clean, df_non_ph_geo_clean], axis=0, ignore_index=True)

# Export final combined file
df_all_geo_clean.to_csv(location_dir/"df_all_geo_clean.csv", index=False)

print("Ph and Non-Ph geocoded files combined.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Manually clean the final concatenated geo file, not upstream files
# Zamboanga is located at latitude 6.91028 and longitude 122.07389.


Ph and Non-Ph geocoded files combined.
Date and Time:  2026-03-10 13:15:13


In [17]:
# 8j. Use folium to create an interactive map for df_all_geo_clean
# df_all_geo_clean → location_map_2026.html
import folium 
import pandas as pd

df_all_geo_clean = pd.read_csv("location_dir/df_all_geo_clean.csv")
# Create a map centered on the Philippines
m = folium.Map(location=[12.8797, 121.7740], zoom_start=6)

# Add markers for each city
for _, row in df_all_geo_clean.iterrows():

    # Choose the correct label
    label = row["city_ph"] if row["country"] == "Philippines" else row["city_non_ph"]

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=label,
        tooltip=label,
        zoom_start  = 6,
        tiles = "CartoDB Positron"
    ).add_to(m)

# Display the map  (Optional - need to restart after Powershell command.))
# m   

# 8k. Using m.save to save the map
m.save("location_dir/location_map_2026.html")

In [15]:
# Visually inspect anomalies
# Replace erroneous latitude and longitude for : 
#   Zamboanga, 6.91028, 122.07389
#   Quezon province,  13.9358, 121.6129
#   Egypt, Alexandria, 31.205753,29.924526
#   United States,Alexandria,1,38.8048,-77.0469
#   Peru,Not Applicable,Pasco,1,-10.6835926,-76.2561123


##### STOP HERE: Manually clean df_all_geo_clean.csv here. 
- Delete previous map.html file.
- Then re-run folium.

# 9. Export data mart sql versions
- df_single_with_grps merge with all df_exploded as list

In [18]:
# 9a. List of all exploded dfs
import pandas as pd
import os
from pathlib import Path
import numpy as np

# Retrieve all the csv outputs
csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)


exploded_filepaths = [
    x for x in csv_outputs_dir.iterdir()
    if x.name.endswith("_exploded_cleaned.csv")
]

exploded_filepaths

[WindowsPath('csv_outputs_dir/ai_tools_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/attended_inperson_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/attended_online_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/busint_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/cloudplat_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/dep_init_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/digitools_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/generaltools_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/hardware_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/ingestion_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/orchest_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/otherfb_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/restofrole_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/spneeds_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/storage_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/succes

In [24]:
# 9b. Dictionary of all  df_exploded files
# Key is {col} file, value is a dataframe
dfs_exploded = {}

for path in exploded_filepaths:
    col = path.stem.replace("_exploded_cleaned", "")
    df = pd.read_csv(path)
    dfs_exploded[col] = df

dfs_exploded.keys()

dict_keys(['ai_tools', 'attended_inperson', 'attended_online', 'busint', 'cloudplat', 'dep_init', 'digitools', 'generaltools', 'hardware', 'ingestion', 'orchest', 'otherfb', 'restofrole', 'spneeds', 'storage', 'successmethod', 'transform', 'volunteer', 'warehs', 'whatused'])

In [27]:
dfs_exploded["ai_tools"].head()

,resp_id,ai_tools
0,1,Chatgpt
1,2,Chatgpt
2,2,Copilot
3,3,Chatgpt
4,3,Copilot


In [28]:
# 9c. Combine everything into duckdb with df_single_with_grps

import duckdb
con = duckdb.connect("survey2026.duckdb")

#  Store all multi-response dfs as tables
for name, df in dfs_exploded.items():
    con.register("tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM tmp")

# Store df_single_with_grps
df_single_with_grps = pd.read_csv(csv_outputs_dir/"df_single_with_grps.csv")
con.register("tmp", df_single_with_grps)
con.execute("CREATE OR REPLACE TABLE df_single_with_grps AS SELECT * FROM tmp")


# Retrieve the combined ph and non ph geo coded file 
location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)

# Store df_all_geo_clean
df_all_geo_clean = pd.read_csv(location_dir/"df_all_geo_clean.csv")
con.register("tmp", df_all_geo_clean)
con.execute("CREATE OR REPLACE TABLE df_all_geo_clean AS SELECT * FROM tmp")

# Inspect all tables
con.execute("SHOW TABLES").df()



,name
0,ai_tools
1,attended_inperson
2,attended_online
3,busint
4,cloudplat
5,dep_init
6,df_all_geo_clean
7,df_single_with_grps
8,digitools
9,generaltools


In [ ]:
# 9d. close the connection
con.close()

# Log
import datetime
print("Duckdb created. Connection closed.")
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

In [32]:
# Log
import datetime
print("Duckdb created. Connection closed.")
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Duckdb created. Connection closed.
Date and Time: 2026-03-10 14:42:55


In [36]:
#9e. Create a sqlite version for side explorations
# A lot smaller than duckdb since it has no metadata

import duckdb
con = duckdb.connect("survey2026.duckdb")

con.execute("DETACH DATABASE IF EXISTS sqlite_db")
con.execute("ATTACH 'survey2026.sqlite' AS sqlite_db (TYPE sqlite)")
tables = con.execute("SHOW TABLES").fetchall()
for (table_name,) in tables:
    con.execute(f"DROP TABLE IF EXISTS sqlite_db.{table_name}")
    con.execute(f"""
        CREATE TABLE sqlite_db.{table_name} AS
        SELECT * FROM {table_name}
    """)
con.close()

# Log
print("Sqlite created. Connection closed.")
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Sqlite created. Connection closed.
Date and Time: 2026-03-10 14:47:53


# 10.  Export data mart excel version for_tableau.xlsx file

In [37]:
import pandas as pd
from pathlib import Path
import shutil

# Directories
csv_outputs_dir = Path("csv_outputs_dir")
location_dir = Path("location_dir")

final_outputs_dir = Path("final_outputs_dir")
final_outputs_dir.mkdir(exist_ok=True)

# 1. Identify final csv files
single_file = csv_outputs_dir / "df_single_with_grps.csv"
geo_file = location_dir / "df_all_geo_clean.csv"
exploded_files = list(csv_outputs_dir.glob("*_exploded_cleaned.csv"))

files_to_copy = [single_file, geo_file] + exploded_files

# 2. Copy final csv files to final_outputs
for f in files_to_copy:
    if f.exists():
        shutil.copy(f, final_outputs_dir / f.name)
    else:
        print(f"Missing file:", f)

# 3. Build Excel workbook from the final CSVs
xlsx_path = final_outputs_dir / "for_tableau.xlsx"

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    for f in files_to_copy:
        if f.exists():
            df = pd.read_csv(f)
            sheet_name = f.stem[:31]  # Excel sheet name limit
            df.to_excel(writer, sheet_name=sheet_name, index=False)

print("for_tableau.xlsx created in final_outputs directory.")


for_tableau.xlsx created in final_outputs directory.
